# Download and Prepare Coat Shopping Dataset

This notebook downloads the Coat Shopping dataset from Cornell and converts it to the format needed for our experiments.

**Dataset Source:** https://www.cs.cornell.edu/~schnabts/mnar/

**Dataset Info:**
- ~290 users, ~300 items
- train.ascii: Self-selected (biased) ratings
- test.ascii: Randomized (unbiased) ratings (~4,640 ratings)


In [15]:
# Import libraries
import urllib.request
import os
import numpy as np
import pandas as pd
from pathlib import Path


In [16]:
# Create data directory
data_dir = Path('../data/coat_data')
data_dir.mkdir(parents=True, exist_ok=True)

# Check if files are in subdirectory (coat folder)
coat_subdir = data_dir / 'coat'
if coat_subdir.exists():
    print(f"Found files in subdirectory: {coat_subdir.absolute()}")
    # Use the subdirectory if it exists
    train_path = coat_subdir / 'train.ascii'
    test_path = coat_subdir / 'test.ascii'
else:
    print(f"Data directory: {data_dir.absolute()}")
    train_path = data_dir / 'train.ascii'
    test_path = data_dir / 'test.ascii'


Found files in subdirectory: c:\Users\kanam\OneDrive\Desktop\pranathi\pranathi\Notebooks\..\data\coat_data\coat


In [17]:
# Check if files already exist (either in main dir or subdirectory)
if train_path.exists() and test_path.exists():
    print("✓ Dataset files already found!")
    print(f"  Train: {train_path}")
    print(f"  Test: {test_path}")
    print("  Skipping download - files are ready!")
else:
    # Download Coat dataset files
    base_url = "https://www.cs.cornell.edu/~schnabts/mnar/"
    files = ['train.ascii', 'test.ascii']
    
    print("Downloading Coat Shopping dataset...")
    print("="*60)
    
    for filename in files:
        url = base_url + filename
        local_path = data_dir / filename
        
        if local_path.exists():
            print(f"✓ {filename} already exists, skipping download")
        else:
            try:
                print(f"Downloading {filename}...")
                urllib.request.urlretrieve(url, str(local_path))
                print(f"✓ Successfully downloaded {filename}")
            except Exception as e:
                print(f"✗ Error downloading {filename}: {e}")
                print(f"  Please manually download from: {url}")
                print(f"  And save to: {local_path}")
    
    print("="*60)
    print("Download complete!")


✓ Dataset files already found!
  Train: ..\data\coat_data\coat\train.ascii
  Test: ..\data\coat_data\coat\test.ascii
  Skipping download - files are ready!


In [18]:
def load_coat_ascii(filepath):
    """
    Load Coat dataset ASCII file and convert to DataFrame
    
    Format: ASCII matrix where:
    - Rows = users
    - Columns = items
    - Values = ratings (0 = missing)
    """
    print(f"\nLoading {filepath.name}...")
    
    # Read the ASCII file
    matrix = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.strip():  # Skip empty lines
                row = [float(x) for x in line.strip().split()]
                matrix.append(row)
    
    matrix = np.array(matrix)
    print(f"  Matrix shape: {matrix.shape} (users x items)")
    
    # Convert to DataFrame format (userId, itemId, rating)
    ratings_list = []
    n_users, n_items = matrix.shape
    
    for user_idx in range(n_users):
        for item_idx in range(n_items):
            rating = matrix[user_idx, item_idx]
            if rating > 0:  # Only include non-zero ratings
                ratings_list.append({
                    'userId': user_idx,
                    'itemId': item_idx,
                    'rating': rating
                })
    
    df = pd.DataFrame(ratings_list)
    print(f"  Total ratings: {len(df)}")
    print(f"  Unique users: {df['userId'].nunique()}")
    print(f"  Unique items: {df['itemId'].nunique()}")
    
    return df


In [19]:
# Load and convert ASCII files to CSV
# Use the paths from Cell 2 (which point to the coat subdirectory)
# If coat subdirectory exists, use those paths; otherwise use main directory
if coat_subdir.exists():
    train_path = coat_subdir / 'train.ascii'
    test_path = coat_subdir / 'test.ascii'
    csv_dir = coat_subdir  # Save CSVs in the same folder
else:
    train_path = data_dir / 'train.ascii'
    test_path = data_dir / 'test.ascii'
    csv_dir = data_dir

if train_path.exists() and test_path.exists():
    print("Converting ASCII files to CSV format...")
    print("="*60)
    
    # Load train (biased) and test (unbiased) data
    train_df = load_coat_ascii(train_path)
    test_df = load_coat_ascii(test_path)
    
    # Save as CSV
    train_csv = csv_dir / 'train.csv'
    test_csv = csv_dir / 'test.csv'
    
    train_df.to_csv(train_csv, index=False)
    test_df.to_csv(test_csv, index=False)
    
    print("\n" + "="*60)
    print("Dataset preparation complete!")
    print(f"Train CSV: {train_csv}")
    print(f"Test CSV: {test_csv}")
    print("\nDataset Statistics:")
    print(f"Train (biased) - Users: {train_df['userId'].nunique()}, Items: {train_df['itemId'].nunique()}, Ratings: {len(train_df)}")
    print(f"Test (unbiased) - Users: {test_df['userId'].nunique()}, Items: {test_df['itemId'].nunique()}, Ratings: {len(test_df)}")
    
    # Also create a combined file for basic NCF experiments
    # Combine train and test for full dataset
    combined_df = pd.concat([train_df, test_df], ignore_index=True)
    combined_csv = csv_dir / 'coat_combined.csv'
    combined_df.to_csv(combined_csv, index=False)
    print(f"\nCombined CSV (for basic NCF): {combined_csv}")
    print(f"Combined - Users: {combined_df['userId'].nunique()}, Items: {combined_df['itemId'].nunique()}, Ratings: {len(combined_df)}")
else:
    print("⚠ Warning: Some files are missing. Please download manually if needed.")


Converting ASCII files to CSV format...

Loading train.ascii...
  Matrix shape: (290, 300) (users x items)
  Total ratings: 6960
  Unique users: 290
  Unique items: 300

Loading test.ascii...
  Matrix shape: (290, 300) (users x items)
  Total ratings: 4640
  Unique users: 290
  Unique items: 300

Dataset preparation complete!
Train CSV: ..\data\coat_data\coat\train.csv
Test CSV: ..\data\coat_data\coat\test.csv

Dataset Statistics:
Train (biased) - Users: 290, Items: 300, Ratings: 6960
Test (unbiased) - Users: 290, Items: 300, Ratings: 4640

Combined CSV (for basic NCF): ..\data\coat_data\coat\coat_combined.csv
Combined - Users: 290, Items: 300, Ratings: 11600


## Verification: Check if this matches the Coat Shopping Dataset description

**Expected characteristics:**
- ~290 users
- ~300 items
- ~4,640 ratings in randomized (test) portion
- Both biased (train) and unbiased (test) ratings


In [20]:
# Verify dataset characteristics match expected Coat Shopping dataset
# Use the correct paths (from coat subdirectory if it exists)
if coat_subdir.exists():
    train_path = coat_subdir / 'train.ascii'
    test_path = coat_subdir / 'test.ascii'
else:
    train_path = data_dir / 'train.ascii'
    test_path = data_dir / 'test.ascii'

if train_path.exists() and test_path.exists():
    print("="*60)
    print("VERIFICATION: Checking if this matches Coat Shopping Dataset")
    print("="*60)
    
    # Check dimensions
    train_matrix = []
    test_matrix = []
    
    with open(train_path, 'r') as f:
        for line in f:
            if line.strip():
                train_matrix.append([float(x) for x in line.strip().split()])
    
    with open(test_path, 'r') as f:
        for line in f:
            if line.strip():
                test_matrix.append([float(x) for x in line.strip().split()])
    
    train_matrix = np.array(train_matrix)
    test_matrix = np.array(test_matrix)
    
    n_users_train, n_items_train = train_matrix.shape
    n_users_test, n_items_test = test_matrix.shape
    
    # Count non-zero ratings
    train_ratings = np.sum(train_matrix > 0)
    test_ratings = np.sum(test_matrix > 0)
    
    print(f"\nDataset Characteristics:")
    print(f"  Train matrix: {train_matrix.shape} (users x items)")
    print(f"  Test matrix: {test_matrix.shape} (users x items)")
    print(f"  Train ratings (biased): {train_ratings}")
    print(f"  Test ratings (unbiased): {test_ratings}")
    
    print(f"\nExpected Coat Shopping Dataset:")
    print(f"  Users: ~290")
    print(f"  Items: ~300")
    print(f"  Test ratings: ~4,640")
    
    print(f"\n" + "="*60)
    print("VERIFICATION RESULTS:")
    print("="*60)
    
    # Check if matches (with some tolerance)
    users_match = (abs(n_users_train - 290) < 20) or (abs(n_users_test - 290) < 20)
    items_match = (abs(n_items_train - 300) < 20) or (abs(n_items_test - 300) < 20)
    test_ratings_match = abs(test_ratings - 4640) < 500
    
    if users_match and items_match and test_ratings_match:
        print("✓ MATCHES Coat Shopping Dataset characteristics!")
        print("  This appears to be the correct dataset.")
    else:
        print("⚠ WARNING: Characteristics don't exactly match expected values.")
        print("  Please verify this is the Coat Shopping dataset.")
        if not users_match:
            print(f"    - User count mismatch: got {n_users_train}/{n_users_test}, expected ~290")
        if not items_match:
            print(f"    - Item count mismatch: got {n_items_train}/{n_items_test}, expected ~300")
        if not test_ratings_match:
            print(f"    - Test ratings mismatch: got {test_ratings}, expected ~4,640")
    
    print("="*60)
else:
    print("⚠ Cannot verify - files not found. Please download first.")


VERIFICATION: Checking if this matches Coat Shopping Dataset

Dataset Characteristics:
  Train matrix: (290, 300) (users x items)
  Test matrix: (290, 300) (users x items)
  Train ratings (biased): 6960
  Test ratings (unbiased): 4640

Expected Coat Shopping Dataset:
  Users: ~290
  Items: ~300
  Test ratings: ~4,640

VERIFICATION RESULTS:
✓ MATCHES Coat Shopping Dataset characteristics!
  This appears to be the correct dataset.


In [21]:
# Display sample data
# Use the correct paths
if coat_subdir.exists():
    train_path = coat_subdir / 'train.ascii'
    test_path = coat_subdir / 'test.ascii'
else:
    train_path = data_dir / 'train.ascii'
    test_path = data_dir / 'test.ascii'

if train_path.exists() and test_path.exists():
    print("\nSample Train Data (biased):")
    print(train_df.head(10))
    print("\nSample Test Data (unbiased):")
    print(test_df.head(10))
    
    print("\nRating distributions:")
    print("\nTrain ratings:")
    print(train_df['rating'].value_counts().sort_index())
    print("\nTest ratings:")
    print(test_df['rating'].value_counts().sort_index())



Sample Train Data (biased):
   userId  itemId  rating
0       0      72     2.0
1       0     136     2.0
2       0     150     3.0
3       0     171     3.0
4       0     188     3.0
5       0     220     3.0
6       0     227     5.0
7       0     228     4.0
8       0     234     3.0
9       0     235     4.0

Sample Test Data (unbiased):
   userId  itemId  rating
0       0      12     4.0
1       0      17     3.0
2       0      74     4.0
3       0      78     2.0
4       0      92     2.0
5       0     104     4.0
6       0     127     4.0
7       0     128     3.0
8       0     133     3.0
9       0     145     2.0

Rating distributions:

Train ratings:
rating
1.0    1901
2.0    1437
3.0    1717
4.0    1275
5.0     630
Name: count, dtype: int64

Test ratings:
rating
1.0    1879
2.0     899
3.0    1002
4.0     641
5.0     219
Name: count, dtype: int64
